# Is Dar es Salaam warmer than forty years ago?

**UDSM / MATH4SDG industry bootcamp · optional backup investigation**

**Scientific question:** Has the annual average temperature recorded at Dar es Salaam Airport increased from 1986 through 2025? How confident should we be in a conclusion drawn from the available data?

This is an *investigation*, not an exercise with a predetermined answer. Document your choices, gaps and limitations. It develops ideas from the [MAT102 Meteostat notebooks](https://github.com/mbr085/V26MAT102) on exploration, regression, seasonal variation and hypothesis testing.

**Data-warning:** The notebook works offline with *synthetic, artificially generated temperatures* unless you have deliberately prepared a Meteostat station CSV. Synthetic values are **not observations of Tanzania's climate** and must never be presented as findings about actual warming.

**Licensing:** Code cells [MIT](../LICENSE); teaching prose [CC BY 4.0](../LICENSE-CONTENT.md); generated synthetic values [CC0](../DATA_LICENSES.md). Real Meteostat and partner data have separate terms.

## 1. Define the quantity we want to estimate

An individual hot day is not a climate trend. We will initially study the least-squares slope of **annual temperature summaries** in °C/decade, and separately compare the first ten and last ten years.

Discuss: Is a single airport station representative of the city? What would it take to extend our conclusions to the entire country? Why do seasonal coverage and station history matter?

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from tools.generate_synthetic_temperature import generate
from tools.temperature_methods import annual_means, estimate_trend

REAL = ROOT / "data" / "private" / "meteostat_63894_1986_2025.csv"
if REAL.exists():
    data = pd.read_csv(REAL, parse_dates=["date"])
    SOURCE = "METEOSTAT STATION ARCHIVE (verify metadata and provenance)"
else:
    data = generate()
    SOURCE = "SYNTHETIC DATA ONLY — NO CLIMATE EVIDENCE"
print(SOURCE)
data = data.sort_values("date").drop_duplicates("date", keep="last")
data["temp"] = pd.to_numeric(data["temp"], errors="coerce")
data = data.set_index("date").loc["1986":"2025"]
if data.empty:
    raise ValueError("The selected dataset contains no observations for 1986–2025.")
print("Rows:", len(data), "| nonmissing daily temperatures:", data["temp"].notna().sum())
data.head()

## 2. Look for gaps before analysing the numbers

A previous observation carried forward is **not** a new temperature measurement. Never silently replace a whole missing season using `.ffill()`. What could happen to an apparent trend if data from warmer months were missing more often in earlier years?

In [ ]:
daily = data["temp"].asfreq("D")
coverage = daily.notna().groupby(daily.index.year).mean()
print("Missing days:", int(daily.isna().sum()), "out of", len(daily))
print("Lowest coverage years:")
print(coverage.nsmallest(8).round(3))
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
daily.iloc[:730].plot(ax=axes[0], title="First two years (gaps left unfilled)")
axes[0].set_ylabel("°C")
coverage.plot(ax=axes[1], marker="o", title="Fraction of days with temperature by year")
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Coverage")
plt.tight_layout()
plt.show()

## 3. Seasonal variation

First calculate mean temperature for each **calendar month**. Does seasonality look large compared with the long-term change you are trying to detect? For an advanced model, MAT102 also studied sine/cosine seasonal predictors.

In [ ]:
seasonal = daily.groupby(daily.index.month).mean()
ax = seasonal.plot(marker="o", figsize=(8, 3), title="Typical seasonal temperature")
ax.set(xlabel="Calendar month", ylabel="°C")
plt.show()

## 4. Annual summaries and model choice

To reduce bias from seasonal gaps, require **at least 20 observed days in each calendar month**, then average the twelve monthly means with equal weight. If any month is inadequate, exclude that year and report the exclusion. This rule is conservative, not universally correct; consider sensitivity to it.

We estimate a descriptive trend

$$T_y = \alpha + \beta (y-y_0) + \varepsilon_y.$$

The slope $10\beta$ has units °C/decade.

In [ ]:
annual, monthly_table = annual_means(daily, min_days_per_month=20)
print(f"Eligible years: {len(annual)} of {len(monthly_table)}")
print("Excluded:", sorted(set(monthly_table.index) - set(annual.index)))
if len(annual) < 15:
    raise ValueError("Too few complete years: investigate a different station, variable, or time period.")
ax = annual.plot(marker="o", figsize=(10, 4), title="Annual mean of adequately observed months")
ax.set(xlabel="Year", ylabel="°C")
plt.show()

## 5. Regression from the inside

The underlying calculation minimizes $\sum_i (y_i-\alpha-\beta t_i)^2$. Ask an AI assistant to explain the design matrix and the roles of its two columns. Verify the answer by constructing the matrix yourself, then use `np.linalg.lstsq` and independently compare with SciPy or scikit-learn. The accompanying functions make that cross-check.

In [ ]:
results = estimate_trend(annual, maxlags=3)
print(f"Descriptive slope: {results['slope_c_decade']:.3f} °C/decade")
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(annual.index, annual.values, s=24, label="Annual summaries")
ax.plot(results["years"], results["fitted"], label="Least-squares line")
ax.set(xlabel="Year", ylabel="°C", title="Station-specific descriptive trend")
ax.legend()
plt.show()

## 6. Statistical uncertainty versus confidence in a conclusion

A conventional regression slope interval assumes an appropriate error model. Annual weather summaries may remain correlated. We compare a standard least-squares confidence interval with a heteroskedasticity/autocorrelation-consistent (HAC, Newey–West style) interval, using three annual lags as a **sensitivity choice**, not a universal correction.

A 95% frequentist confidence interval is **not** a 95% probability that the fixed true trend lies in the particular computed interval. A small $p$-value is not the probability that the null hypothesis is true. Neither interval can repair a change in station, biased missing months, or an unjustified linear model.

In [ ]:
print("Slope (°C/decade):", round(results["slope_c_decade"], 3))
print("Conventional 95% interval:", np.round(results["conventional_ci_c_decade"], 3))
print("HAC 95% interval, annual maxlags=3:", np.round(results["hac_ci_c_decade"], 3))
print("Lag-one correlation of annual residuals:", round(results["residual_lag1"], 3))
print("Reminder:", SOURCE)

## 7. “Warmer than forty years ago”: a different estimand

Compare **1986–1995** with **2016–2025**, provided at least eight eligible years occur in each decade. This answers a different question from the linear slope; do the results tell a compatible story? Do not treat ten consecutive annual temperatures as ten independent samples without inspection.

In [ ]:
early = annual.loc[1986:1995]
late = annual.loc[2016:2025]
print("Early decade eligible years:", len(early), "/ 10")
print("Late decade eligible years:", len(late), "/ 10")
if len(early) >= 8 and len(late) >= 8:
    print(f"Recent minus early decade mean: {late.mean() - early.mean():.3f} °C")
else:
    print("Insufficient coverage for a reliable decade comparison.")

## 8. Challenge the conclusion

1. Re-run the analysis with 15 and 25 observed days per month. Which years are included?
2. Change the first year to 1991, 1996 and 2001. How stable is the trend?
3. Plot the residuals; do you see a jump, curved pattern or serial dependence?
4. Ask AI to identify **three hidden assumptions** in the code. Independently check each proposed issue.
5. If using observations, compare with TMA documentation or another station only after checking station histories and data completeness.

**Group report:** In five minutes explain your precise claim, data and provenance, method, uncertainty, sensitivity, limitations, and one concrete next step. It is acceptable to conclude that the evidence is insufficient. Do not report synthetic outputs as findings about Tanzania.

In [ ]:
from scipy import stats

checks = []
for first in [1986, 1991, 1996, 2001]:
    part = annual.loc[first:2025]
    if len(part) >= 15:
        slope = stats.linregress(part.index.to_numpy(dtype=float), part.values).slope * 10
        checks.append((first, len(part), round(slope, 3)))
print(pd.DataFrame(checks, columns=["start_year", "eligible_years", "slope_C_per_decade"]).to_string(index=False))
plt.figure(figsize=(9, 3))
plt.plot(results["years"], annual.values - results["fitted"], marker="o")
plt.axhline(0, linestyle="--")
plt.title("Residuals: look for temporal structure or discontinuity")
plt.xlabel("Year")
plt.ylabel("°C")
plt.show()

### Optional extension

Fit a model to monthly temperature anomalies or to daily values with sine/cosine seasonal terms, using **time-aware validation**. Compare different station records only if provenance and coverage allow it. Sophisticated ML tools do not by themselves establish causal attribution or make a single station representative of regional climate.

**Further reading:** [Meteostat daily API](https://dev.meteostat.net/python/api/meteostat.daily) and [Meteostat licensing](https://dev.meteostat.net/license).